Test set ek final exam hai. Is par model ko ek hi baar jaanchna hai, aur result dekh kar model badalna nahi hai. Agar test score dekh kar C ya features tweak karoge, to test set bhi dheere-dheere “training” ka hissa ban jaata hai aur score jhootha ho jaata hai. Tuning ka kaam CV par hota hai (Day 69, 84).

1. Model aur data load karo:

In [16]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

model = joblib.load('models/logreg_pipeline.joblib')    # Day 66 ki saved pipeline (scaler andar hai)

X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(X_test.shape, y_test.shape)

(1057, 8) (1057,)


Test data ko scale karne ki zaroorat nahi. Pipeline ke andar train se seekha hua scaler apne aap lagega (Day 64 ka rule).

2. predict(): hard labels (0 ya 1):

In [ ]:
y_pred = model.predict(X_test)


print(y_pred[:15])
print("Predicted churners:", y_pred.sum(), "of", len(y_pred))
print("Actual churners   :", y_test.sum(), "of", len(y_test))

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
Predicted churners: 559 of 1057
Actual churners   : 598 of 1057


Dono counts ko dekho. Agar class_weight ya SMOTE use kiya hai, to predicted churners aksar actual se zyada honge, kyunki model churn ko zyada pakadne ki koshish karta hai.

3. predict_proba(): probability (zyada informative):

In [18]:
proba = model.predict_proba(X_test)        # shape (n, 2): [P(not churn), P(churn)]
churn_prob = proba[:, 1]

results = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred,
    'churn_prob': churn_prob.round(3)
})
print(results.head(10))

   actual  predicted  churn_prob
0       1          1       0.678
1       1          1       0.587
2       0          1       0.767
3       1          1       0.738
4       0          1       0.939
5       1          1       0.732
6       1          1       0.664
7       1          1       0.853
8       0          1       0.696
9       0          1       0.620


predict() internally yehi karta hai: churn_prob >= 0.5 to 1, warna 0.

4. Verify karo ki predict() = threshold 0.5:

In [19]:
manual = (churn_prob >= 0.5).astype(int)
print("predict() == threshold 0.5:", np.array_equal(manual, y_pred))

predict() == threshold 0.5: True


5. Pehla scoreboard (detail Day 71-74 mein aayegi, aaj sirf nazar daalo):

In [20]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"F1       : {f1_score(y_test, y_pred):.3f}")

majority = y_test.value_counts(normalize=True).max()
print(f"\nNaive baseline accuracy (Day 63): {majority:.3f}")

Accuracy : 0.742
Precision: 0.791
Recall   : 0.739
F1       : 0.764

Naive baseline accuracy (Day 63): 0.566


Accuracy ko akele mat dekho. Baseline se compare karo aur Recall/F1 par dhyan do (Day 56B).

6. Train vs CV vs Test: teeno ka comparison (yeh sabse zaroori check hai):

In [21]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')

print(f"Train F1: {f1_score(y_train, model.predict(X_train)):.3f}")
print(f"CV F1   : {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}")
print(f"Test F1 : {f1_score(y_test, y_pred):.3f}")

Train F1: 0.747
CV F1   : 0.745 +/- 0.010
Test F1 : 0.764


read:

- Test F1 ≈ CV F1: model stable hai, Day 64B ka estimate sahi tha.
- Test F1 CV se kaafi kam: kuch gadbad hai (leakage ya overfitting). Pehle cause dhoondo.
- Test F1 CV se kaafi zyada: suspicious hai, split ya leakage dobara check karo.
- Agar CV ke baad model ko dobara fit karna pade, to cross_val_score ke baad purana fitted model hi chalta hai (cross_val_score uski copy par kaam karta hai).

7. Ek-ek customer ki prediction (Streamlit app ka preview):

In [22]:
one = X_test.iloc[[0]]
p = model.predict_proba(one)[0, 1]
label = model.predict(one)[0]

print(one.T)
print(f"\nChurn probability: {p:.1%}  ->  predicted: {'CHURN' if label == 1 else 'ACTIVE'}  (actual: {y_test.iloc[0]})")

                             0
Recency                    193
Frequency                    2
Monetary_log          5.795602
AvgOrderValue          163.925
Tenure                     498
UniqueProducts              26
Region_Rest of World     False
Region_UK                 True

Churn probability: 67.8%  ->  predicted: CHURN  (actual: 1)


Day 89 mein user form se aaya input bhi isi tarah ek-row DataFrame banega aur isi tarah predict hoga. Column names aur order wahi rehne chahiye jo training mein the.

8. Galat predictions ko dekho:

In [23]:
check = X_test.copy()
check['actual'] = y_test.values
check['pred'] = y_pred
check['prob'] = churn_prob

wrong = check[check['actual'] != check['pred']]
print(f"Wrong predictions: {len(wrong)} of {len(check)}")

fn = check[(check['actual'] == 1) & (check['pred'] == 0)]   # churn miss hua
fp = check[(check['actual'] == 0) & (check['pred'] == 1)]   # galat alarm
print(f"Missed churners (False Negative): {len(fn)}")
print(f"False alarms (False Positive)   : {len(fp)}")
print(fn.describe().loc[['mean']].round(2))
print(fp.describe().loc[['mean']].round(2))

Wrong predictions: 273 of 1057
Missed churners (False Negative): 156
False alarms (False Positive)   : 117
      Recency  Frequency  Monetary_log  AvgOrderValue  Tenure  UniqueProducts  \
mean    89.42       6.04          7.21         374.87  388.61           85.42   

      actual  pred  prob  
mean     1.0   0.0  0.35  
      Recency  Frequency  Monetary_log  AvgOrderValue  Tenure  UniqueProducts  \
mean   287.68       2.56          6.17         310.51  439.96           38.22   

      actual  pred  prob  
mean     0.0   1.0  0.68  


Business mein dono galtiyon ki cost alag hoti hai. Churner miss hua to customer chala gaya. False alarm hua to ek discount faltu gaya. Confusion matrix ka formal detail Day 71 mein hai.

9. Threshold ka effect (ek jhalak):

In [24]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_t = (churn_prob >= t).astype(int)
    rows.append({'threshold': t,
                 'flagged': pred_t.sum(),
                 'precision': precision_score(y_test, pred_t, zero_division=0),
                 'recall': recall_score(y_test, pred_t)})
print(pd.DataFrame(rows).round(3))

   threshold  flagged  precision  recall
0        0.3      809      0.685   0.926
1        0.4      686      0.739   0.848
2        0.5      559      0.791   0.739
3        0.6      441      0.812   0.599
4        0.7      321      0.838   0.450


Threshold kam karne par zyada customers flag hote hain (recall badhta hai, precision ghatta hai). Yeh dikhane ke liye hai, best threshold test set se mat chuno. Agar threshold badalna ho to validation/CV se chuno (Day 74 mein ROC par detail).

10. Predictions save karo (Day 70 comparison ke liye):

In [25]:
results.to_csv('../data/processed/logreg_test_predictions.csv', index=False)

practice questions:

1. Apne results table se 5 customers chuno jinki churn probability sabse zyada hai. Unke features dekh kar likho: kya yeh business sense banata hai (kam Frequency, zyada Recency)?

In [26]:
# Highest churn probability wale top 5 customers
top_5_churners = check.sort_values(by='prob', ascending=False).head(5)
print(top_5_churners[['Recency', 'Frequency', 'Monetary_log', 'actual', 'pred', 'prob']])

     Recency  Frequency  Monetary_log  actual  pred      prob
220      574          1      0.000000       1     1  0.993832
602      540          2      9.540891       1     1  0.977694
637      443          1      1.987874       1     1  0.969134
803      633          1      4.118712       1     1  0.963851
843      599          1      4.126489       1     1  0.956976


2. Apne shabdon mein likho: predict() aur predict_proba() mein kya fark hai, aur business ke liye probability zyada useful kyun hai?